In [ ]:
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [ ]:
spark = (
    SparkSession.builder
    .appName("nyc-yellow-taxi-2025")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Spark version: 4.0.3


## Import data from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Find NYC Yellow Taxi 2025 Parquet files

In [ ]:
import glob
import os

files = glob.glob("/content/drive/MyDrive/**/*.parquet", recursive=True)

taxi_files = sorted([
    f for f in files
    if "yellow_tripdata_2025" in os.path.basename(f).lower()
])

print("Yellow taxi 2025 files found:", len(taxi_files))

for f in taxi_files:
    print(f)

assert len(taxi_files) == 12, "Expected 12 monthly NYC Yellow Taxi 2025 parquet files."

Yellow taxi 2025 files found: 12
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-01.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-02.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-03.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-04.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-05.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-06.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-07.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-08.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-09.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-10.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-11.parquet
/content/drive/MyDrive/NYC taxi/yellow_tripdata_2025-12.parquet


## Per-file validation

In [ ]:
# Reason:
# Before combining files, check that each monthly parquet file is readable,
# has the same schema, and contains rows.

file_validation_logs = []

base_schema = None

for f in taxi_files:
    temp_df = spark.read.parquet(f)
    row_count = temp_df.count()
    col_count = len(temp_df.columns)
    schema_text = temp_df.schema.simpleString()

    if base_schema is None:
        base_schema = schema_text

    schema_match = schema_text == base_schema

    file_validation_logs.append((
        os.path.basename(f),
        row_count,
        col_count,
        schema_match
    ))

file_validation_df = spark.createDataFrame(
    file_validation_logs,
    ["file_name", "row_count", "column_count", "schema_matches_first_file"]
)

file_validation_df.show(20, truncate=False)

assert file_validation_df.filter(F.col("row_count") == 0).count() == 0, "One or more files have 0 rows."
assert file_validation_df.filter(F.col("column_count") != 20).count() == 0, "One or more files do not have 20 columns."
assert file_validation_df.filter(F.col("schema_matches_first_file") == False).count() == 0, "Schema mismatch found."

+-------------------------------+---------+------------+-------------------------+
|file_name                      |row_count|column_count|schema_matches_first_file|
+-------------------------------+---------+------------+-------------------------+
|yellow_tripdata_2025-01.parquet|3475226  |20          |true                     |
|yellow_tripdata_2025-02.parquet|3577543  |20          |true                     |
|yellow_tripdata_2025-03.parquet|4145257  |20          |true                     |
|yellow_tripdata_2025-04.parquet|3970553  |20          |true                     |
|yellow_tripdata_2025-05.parquet|4591845  |20          |true                     |
|yellow_tripdata_2025-06.parquet|4322960  |20          |true                     |
|yellow_tripdata_2025-07.parquet|3898963  |20          |true                     |
|yellow_tripdata_2025-08.parquet|3574091  |20          |true                     |
|yellow_tripdata_2025-09.parquet|4251015  |20          |true                     |
|yel

## Load the full dataset

In [ ]:
trips = spark.read.parquet(*taxi_files)

trips.show(20)
trips.printSchema()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-05-01 00:07:06|  2025-05-01 00:24:15|              1|          3.7|         1|                 N|         140|    

## Start cleaning log

In [ ]:
cleaning_logs = []

cleaning_logs.append(("Step 1", "Loaded 12 NYC Yellow Taxi 2025 parquet files from Google Drive."))
cleaning_logs.append(("Step 2", "Validated each monthly file for row count, column count, and schema consistency."))
cleaning_logs.append(("Step 3", "Checked schema and sample rows using printSchema() and show()."))

print("Cleaning log started")

Cleaning log started


## Count raw rows and columns

In [ ]:
raw_count = trips.count()

print("Raw rows:", raw_count)
print("Total columns:", len(trips.columns))

cleaning_logs.append(("Step 4", f"Raw dataset has {raw_count} rows and {len(trips.columns)} columns."))

Raw rows: 48722602
Total columns: 20


## Full missing-value check for all 20 columns

In [ ]:
# This checks every column, not only selected important columns.

missing_values = trips.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in trips.columns
])

missing_values.show(truncate=False)

cleaning_logs.append(("Step 5", "Checked missing/null values for all 20 columns before cleaning."))

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|0       |0                   |0                    |11611894       |0            |11611894  |11611894          |0           |0   

## Basic numeric summary before cleaning

In [ ]:
# Reason:
# Summary statistics help us set realistic filter thresholds
# instead of blindly removing rows.

numeric_cols = [
    "VendorID",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee"
]

trips.select(numeric_cols).describe().show(truncate=False)

cleaning_logs.append(("Step 6", "Generated describe() summary for numeric columns before deciding cleaning thresholds."))

+-------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+---------------------+------------------+--------------------+-------------------+------------------+
|summary|VendorID          |passenger_count   |trip_distance    |RatecodeID        |PULocationID     |DOLocationID      |payment_type      |fare_amount       |extra             |mta_tax           |tip_amount       |tolls_amount      |improvement_surcharge|total_amount      |congestion_surcharge|Airport_fee        |cbd_congestion_fee|
+-------+------------------+------------------+-----------------+------------------+-----------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+---------------------+------------------+--------------------+-------------------+------------

## Duplicate row check

In [ ]:
# Exact duplicate check.
# Note: This can take time because the dataset is large.

duplicate_rows = raw_count - trips.dropDuplicates().count()

print("Duplicate rows:", duplicate_rows)

cleaning_logs.append(("Step 7", f"Checked exact duplicate rows. Duplicate rows found: {duplicate_rows}."))

Duplicate rows: 1


## Check invalid category values before cleaning

In [ ]:
# TLC valid values used:
# VendorID: 1, 2
# RatecodeID: 1 to 6
# payment_type: 1 to 6

print("VendorID distribution:")
trips.groupBy("VendorID").count().orderBy("VendorID").show(truncate=False)

print("RatecodeID distribution:")
trips.groupBy("RatecodeID").count().orderBy("RatecodeID").show(truncate=False)

print("payment_type distribution:")
trips.groupBy("payment_type").count().orderBy("payment_type").show(truncate=False)

invalid_vendorid_count = trips.filter(~F.col("VendorID").isin([1, 2]) | F.col("VendorID").isNull()).count()
invalid_ratecode_count = trips.filter(~F.col("RatecodeID").between(1, 6) | F.col("RatecodeID").isNull()).count()
invalid_payment_type_count = trips.filter(~F.col("payment_type").between(1, 6) | F.col("payment_type").isNull()).count()

print("Invalid VendorID rows:", invalid_vendorid_count)
print("Invalid RatecodeID rows:", invalid_ratecode_count)
print("Invalid payment_type rows:", invalid_payment_type_count)

cleaning_logs.append(("Step 8", "Checked invalid values for VendorID, RatecodeID, and payment_type before cleaning."))

VendorID distribution:
+--------+--------+
|VendorID|count   |
+--------+--------+
|1       |9586873 |
|2       |38575846|
|6       |23982   |
|7       |535901  |
+--------+--------+

RatecodeID distribution:
+----------+--------+
|RatecodeID|count   |
+----------+--------+
|NULL      |11611894|
|1         |34292985|
|2         |1297934 |
|3         |150692  |
|4         |116228  |
|5         |448421  |
|6         |51      |
|99        |804397  |
+----------+--------+

payment_type distribution:
+------------+--------+
|payment_type|count   |
+------------+--------+
|0           |11611894|
|1           |31054000|
|2           |4654345 |
|3           |308147  |
|4           |1094213 |
|5           |3       |
+------------+--------+

Invalid VendorID rows: 559883
Invalid RatecodeID rows: 12416291
Invalid payment_type rows: 11611894


## Check negative amount columns before cleaning

In [ ]:
amount_cols = [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee"
]

negative_amount_check = trips.select([
    F.count(F.when(F.col(c) < 0, c)).alias(c)
    for c in amount_cols
])

negative_amount_check.show(truncate=False)

cleaning_logs.append(("Step 9", "Checked negative amount values before cleaning, including Airport_fee."))

+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|fare_amount|extra |mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2848620    |414216|762381 |1693      |72894       |804443               |973721      |635917              |166785     |509663            |
+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+



## Per-column cleaning notes before cleaning

| Column / Group | Issue checked | Cleaning rule | Why |
|---|---|---|---|
| tpep_pickup_datetime, tpep_dropoff_datetime | Missing time, wrong duration | Drop missing values and keep trip_minutes between 1 and 180 | Trips need valid start/end time for analysis |
| VendorID | Invalid vendor code such as 6/7 | Keep only 1 and 2 | TLC dictionary defines valid vendors as 1 and 2 |
| passenger_count | Missing or unrealistic values | Fill missing as 1, keep 1 to 6 | Avoid unrealistic passenger counts |
| trip_distance | Zero, negative, extreme values | Keep > 0 and <= 100 miles | Removes invalid/extreme trips |
| RatecodeID | Invalid values such as 99 | Keep 1 to 6 | TLC valid range is 1 to 6 |
| PULocationID, DOLocationID | Invalid taxi zone IDs | Keep 1 to 265 | NYC taxi zone IDs are in this range |
| payment_type | Invalid payment code | Keep 1 to 6 | TLC valid payment codes are 1 to 6 |
| fare_amount, total_amount | Zero, negative, extreme values | Keep > 0 and <= 1000 | Removes invalid billing records |
| extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, congestion_surcharge, Airport_fee, cbd_congestion_fee | Null or negative charges | Fill null as 0 and remove negatives | Charges should not be negative for final analysis |
| pickup_hour, day_of_week, pickup_month | Needed for analysis | Create from pickup datetime | Useful for time-based analysis |
| fare_per_mile, tip_percent, is_cbd | Needed for analysis | Create calculated columns | Useful for fare, tip, and congestion analysis |

## Add useful columns and optimize data types

In [ ]:
df = trips

# Cast small-range integer columns to smaller Spark types.
# This improves dtype optimization compared to keeping everything as long/double.

df = (
    df
    .withColumn("VendorID", F.col("VendorID").cast(ByteType()))
    .withColumn("passenger_count", F.col("passenger_count").cast(ByteType()))
    .withColumn("RatecodeID", F.col("RatecodeID").cast(ByteType()))
    .withColumn("PULocationID", F.col("PULocationID").cast(ShortType()))
    .withColumn("DOLocationID", F.col("DOLocationID").cast(ShortType()))
    .withColumn("payment_type", F.col("payment_type").cast(ByteType()))
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime").cast(ByteType()))
    .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime").cast(ByteType()))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime").cast(ByteType()))
    .withColumn(
        "trip_minutes",
        (
            F.unix_timestamp("tpep_dropoff_datetime")
            - F.unix_timestamp("tpep_pickup_datetime")
        ) / 60
    )
    .withColumn("trip_minutes", F.col("trip_minutes").cast(FloatType()))
    .withColumn(
        "fare_per_mile",
        F.when(
            F.col("trip_distance") > 0,
            F.round(F.col("fare_amount") / F.col("trip_distance"), 2)
        )
    )
    .withColumn("fare_per_mile", F.col("fare_per_mile").cast(FloatType()))
    .withColumn(
        "tip_percent",
        F.when(
            F.col("fare_amount") > 0,
            F.round((F.col("tip_amount") / F.col("fare_amount")) * 100, 2)
        ).otherwise(0)
    )
    .withColumn("tip_percent", F.col("tip_percent").cast(FloatType()))
    .withColumn(
        "is_cbd",
        F.when(F.col("cbd_congestion_fee") > 0, 1).otherwise(0).cast(ByteType())
    )
)

df.select(
    "VendorID",
    "passenger_count",
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "pickup_hour",
    "day_of_week",
    "pickup_month",
    "trip_minutes",
    "fare_per_mile",
    "tip_percent",
    "is_cbd"
).show(5)

df.printSchema()

cleaning_logs.append(("Step 10", "Created analysis columns and optimized small integer columns using ByteType/ShortType."))

+--------+---------------+----------+------------+------------+------------+-----------+-----------+------------+------------+-------------+-----------+------+
|VendorID|passenger_count|RatecodeID|PULocationID|DOLocationID|payment_type|pickup_hour|day_of_week|pickup_month|trip_minutes|fare_per_mile|tip_percent|is_cbd|
+--------+---------------+----------+------------+------------+------------+-----------+-----------+------------+------------+-------------+-----------+------+
|       1|              1|         1|         140|         202|           1|          0|          5|           5|       17.15|         4.97|      26.36|     1|
|       2|              1|         1|         234|         161|           1|          0|          5|           5|   6.7166667|         8.35|       50.0|     1|
|       2|              1|         1|         161|         234|           2|          0|          5|           5|        7.95|         6.37|        0.0|     1|
|       2|              1|         1|   

## Clean invalid rows

In [ ]:
clean = df.na.drop(subset=[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "VendorID",
    "RatecodeID",
    "payment_type",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "trip_distance",
    "total_amount",
    "trip_minutes"
])

cleaning_logs.append(("Step 11", "Dropped rows where essential fields were missing."))

# Fill only nullable amount/passenger fields where 0 or safe default is meaningful.
# This does not change negative values; negative values are removed in the filter below.
clean = clean.na.fill({
    "passenger_count": 1,
    "tip_amount": 0,
    "tolls_amount": 0,
    "extra": 0,
    "mta_tax": 0,
    "improvement_surcharge": 0,
    "congestion_surcharge": 0,
    "Airport_fee": 0,
    "cbd_congestion_fee": 0
})

cleaning_logs.append(("Step 12", "Filled safe nullable numeric fields with default values."))

clean = clean.filter(
    (F.col("VendorID").isin([1, 2])) &
    (F.col("RatecodeID").between(1, 6)) &
    (F.col("payment_type").between(1, 6)) &
    (F.col("trip_distance") > 0) &
    (F.col("trip_distance") <= 100) &
    (F.col("fare_amount") > 0) &
    (F.col("fare_amount") <= 1000) &
    (F.col("total_amount") > 0) &
    (F.col("total_amount") <= 1000) &
    (F.col("trip_minutes") > 0) &
    (F.col("trip_minutes") <= 180) &
    (F.col("PULocationID").between(1, 265)) &
    (F.col("DOLocationID").between(1, 265)) &
    (F.col("passenger_count").between(1, 6)) &
    (F.col("extra") >= 0) &
    (F.col("mta_tax") >= 0) &
    (F.col("tip_amount") >= 0) &
    (F.col("tolls_amount") >= 0) &
    (F.col("improvement_surcharge") >= 0) &
    (F.col("congestion_surcharge") >= 0) &
    (F.col("Airport_fee") >= 0) &
    (F.col("cbd_congestion_fee") >= 0)
)

cleaning_logs.append(("Step 13", "Removed invalid rows using range filters for VendorID, RatecodeID, payment_type, distance, fare, duration, location IDs, passenger count, and non-negative amount columns."))

clean.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-----------+-----------+------------+------------+-------------+-----------+------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_hour|day_of_week|pickup_month|trip_minutes|fare_per_mile|tip_percent|is_cbd|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+-----------------

## Cleaning summary

In [ ]:
clean_count = clean.count()
removed_count = raw_count - clean_count
removed_percentage = round((removed_count / raw_count) * 100, 2)

print("Raw rows:", raw_count)
print("Clean rows:", clean_count)
print("Removed rows:", removed_count)
print("Removed percentage:", removed_percentage, "%")

cleaning_logs.append(("Step 14", f"After cleaning, {clean_count} rows remained. Removed {removed_count} rows, which is {removed_percentage}% of raw data."))

Raw rows: 48722602
Clean rows: 34273598
Removed rows: 14449004
Removed percentage: 29.66 %


## Final quality checks and assertions

In [ ]:
bad_rows = clean.filter(
    (F.col("trip_distance") <= 0) |
    (F.col("fare_amount") <= 0) |
    (F.col("total_amount") <= 0) |
    (F.col("trip_minutes") <= 0) |
    (~F.col("VendorID").isin([1, 2])) |
    (~F.col("RatecodeID").between(1, 6)) |
    (~F.col("payment_type").between(1, 6)) |
    (~F.col("PULocationID").between(1, 265)) |
    (~F.col("DOLocationID").between(1, 265)) |
    (~F.col("passenger_count").between(1, 6)) |
    (F.col("Airport_fee") < 0)
).count()

negative_amount_after_clean = clean.select([
    F.count(F.when(F.col(c) < 0, c)).alias(c)
    for c in amount_cols
])

negative_amount_after_clean.show(truncate=False)

print("Bad rows after cleaning:", bad_rows)

assert bad_rows == 0, "Final quality check failed: invalid rows still exist."
assert clean.filter(F.col("Airport_fee") < 0).count() == 0, "Airport_fee still has negative values."
assert clean.filter(~F.col("VendorID").isin([1, 2])).count() == 0, "Invalid VendorID still exists."
assert clean.filter(~F.col("RatecodeID").between(1, 6)).count() == 0, "Invalid RatecodeID still exists."
assert clean.filter(~F.col("payment_type").between(1, 6)).count() == 0, "Invalid payment_type still exists."

cleaning_logs.append(("Step 15", f"Final quality checks completed. Bad rows after cleaning: {bad_rows}. All assertions passed."))

+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|0          |0    |0      |0         |0           |0                    |0           |0                   |0          |0                 |
+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+

Bad rows after cleaning: 0


## Save cleaned data as Parquet

In [ ]:
clean_output_path = "/content/drive/MyDrive/NYC taxi/outputs/cleaned_data"

(
    clean
    .write
    .mode("overwrite")
    .partitionBy("pickup_month")
    .parquet(clean_output_path)
)

print("Cleaned data saved at:")
print(clean_output_path)

# Validate saved parquet by reading it again.
saved_clean = spark.read.parquet(clean_output_path)
saved_count = saved_clean.count()

print("Saved cleaned rows:", saved_count)
print("Clean DataFrame rows:", clean_count)

assert saved_count == clean_count, "Saved parquet row count does not match clean DataFrame row count."

cleaning_logs.append(("Step 16", "Saved cleaned DataFrame as Parquet partitioned by pickup_month and verified saved row count."))

Cleaned data saved at:
/content/drive/MyDrive/NYC taxi/outputs/cleaned_data
Saved cleaned rows: 34273598
Clean DataFrame rows: 34273598


## Compare raw vs cleaned Parquet storage size

In [ ]:
# This is a practical storage-size comparison.
# Exact Spark memory usage depends on partitions/cache, so file-size comparison is easier to verify in Colab.

def folder_size_bytes(folder_path):
    total = 0
    for root, dirs, files_in_dir in os.walk(folder_path):
        for name in files_in_dir:
            full_path = os.path.join(root, name)
            if os.path.isfile(full_path):
                total += os.path.getsize(full_path)
    return total

raw_size_bytes = sum(os.path.getsize(f) for f in taxi_files)
clean_size_bytes = folder_size_bytes(clean_output_path)

raw_size_mb = round(raw_size_bytes / (1024 * 1024), 2)
clean_size_mb = round(clean_size_bytes / (1024 * 1024), 2)

print("Raw parquet size MB:", raw_size_mb)
print("Cleaned parquet size MB:", clean_size_mb)
print("Size difference MB:", round(raw_size_mb - clean_size_mb, 2))

cleaning_logs.append(("Step 17", f"Compared raw parquet size ({raw_size_mb} MB) with cleaned parquet size ({clean_size_mb} MB)."))

Raw parquet size MB: 791.52
Cleaned parquet size MB: 853.4
Size difference MB: -61.88


## Save cleaning logs

In [ ]:
cleaning_log_df = spark.createDataFrame(cleaning_logs, ["step", "description"])

cleaning_log_df.show(truncate=False)

log_output_path = "/content/drive/MyDrive/NYC taxi/outputs/cleaning_logs"

cleaning_log_df.coalesce(1).write.mode("overwrite").option("header", True).csv(log_output_path)

print("Cleaning logs saved at:")
print(log_output_path)

+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|step   |description                                                                                                                                                               |
+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Step 1 |Loaded 12 NYC Yellow Taxi 2025 parquet files from Google Drive.                                                                                                           |
|Step 2 |Validated each monthly file for row count, column count, and schema consistency.                                                                                          |
|Step 3 |Checked schema and sample rows using printSchema() and show().                        

## Cleaning Logs

1. Loaded 12 NYC Yellow Taxi 2025 Parquet files from Google Drive.  
   **Why:** The dataset is large, so PySpark is better than normal pandas.

2. Validated all monthly files one by one.  
   **Why:** To confirm every file is readable, has rows, has 20 columns, and follows the same schema.

3. Checked schema and sample rows.  
   **Why:** To understand column names, data types, and structure.

4. Counted raw rows and columns.  
   **Why:** To compare raw data size with cleaned data size.

5. Checked missing values for all 20 columns.  
   **Why:** Missing values can affect analysis and earlier check skipped some columns.

6. Generated numeric summary using `describe()`.  
   **Why:** To understand min, max, mean, and decide realistic cleaning thresholds.

7. Checked duplicate rows.  
   **Why:** Exact duplicate trips can bias analysis.

8. Checked invalid `VendorID`, `RatecodeID`, and `payment_type` values.  
   **Why:** TLC data dictionary allows only specific category ranges.

9. Checked negative amount columns, including `Airport_fee`.  
   **Why:** Negative charges should not remain in the final cleaned dataset.

10. Added useful columns and optimized data types.  
    **Why:** Time, fare, tip, and congestion columns are useful for dashboard/analysis, and smaller integer types save space.

11. Dropped rows with missing essential fields.  
    **Why:** Trips without date, location, fare, distance, or category values cannot be trusted.

12. Filled safe nullable numeric values.  
    **Why:** Some missing amount fields can be safely treated as 0.

13. Applied final range filters.  
    **Why:** To remove invalid trips, invalid categories, negative charges, and unrealistic values.

14. Counted cleaned rows and removed rows.  
    **Why:** To show the exact cleaning impact.

15. Ran final assertions.  
    **Why:** To prove invalid rows like negative `Airport_fee`, `VendorID=7`, invalid `RatecodeID`, and invalid `payment_type` are removed.

16. Saved cleaned data as Parquet partitioned by `pickup_month`.  
    **Why:** Parquet is efficient for big data, and monthly partitioning makes future analysis faster.

17. Compared raw vs cleaned storage size.  
    **Why:** To show the impact of cleaning and dtype optimization.